# Game Phase Analysis — Where Are Games Being Decided?
**Project 1500**

All statistics computed fresh from game files on every run.
Losses are bucketed by game length (proxy for phase where the game was decided).

In [ ]:
import chess.pgn
import datetime
import glob
import io
import json
import re
from collections import Counter, defaultdict
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np

# ── config ────────────────────────────────────────────────────────────────────
USERNAME       = 'sfink37'
GAMES_DIR      = 'games_sf'
TIME_CLASSES   = {'bullet', 'blitz'}   # e.g. {'rapid'} for jf4bes
EXCLUDE_MONTHS = set()                 # e.g. {'2025_01','2025_02','2025_03'}
# ─────────────────────────────────────────────────────────────────────────────

OPENING_END    = 20   # half-moves (move 10)
MIDDLEGAME_END = 50   # half-moves (move 25)

PHASES = [
    'Opening (<=move 10)',
    'Middlegame (move 11-25)',
    'Endgame (move 26+)',
]

def load_games(username, games_dir, time_classes, exclude_months=set()):
    files = sorted(glob.glob(f'{games_dir}/*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude_months)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') not in time_classes: continue
            white = g.get('white', {})
            black = g.get('black', {})
            if white.get('username','').lower() == username.lower():
                color, my, opp = 'white', white, black
            elif black.get('username','').lower() == username.lower():
                color, my, opp = 'black', black, white
            else:
                continue
            result = my.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient',
                            'timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            games.append({
                'outcome':    outcome,
                'color':      color,
                'end_reason': result,
                'time_class': g.get('time_class'),
                'pgn':        g.get('pgn', ''),
                'eco_url':    m.group(1) if m else '',
                'my_rating':  my.get('rating', 0),
                'opp_rating': opp.get('rating', 0),
                'end_time':   g.get('end_time', 0),
            })
    return sorted(games, key=lambda x: x['end_time'])

def count_halfmoves(pgn_str):
    try:
        game = chess.pgn.read_game(io.StringIO(pgn_str))
        return len(list(game.mainline_moves()))
    except Exception:
        return 0

def assign_phase(hm):
    if hm <= OPENING_END:    return PHASES[0]
    if hm <= MIDDLEGAME_END: return PHASES[1]
    return PHASES[2]

def classify_end(reason):
    if reason == 'checkmated':             return 'Checkmate'
    if reason in ('resigned', 'lose'):     return 'Resignation'
    if reason in ('timeout', 'abandoned'): return 'Timeout'
    return 'Other'

## Step 1 — Load games

In [ ]:
games = load_games(USERNAME, GAMES_DIR, TIME_CLASSES, EXCLUDE_MONTHS)
for g in games:
    g['halfmoves'] = count_halfmoves(g['pgn'])
    g['phase']     = assign_phase(g['halfmoves'])

losses       = [g for g in games if g['outcome'] == 'loss']
total        = len(games)
total_losses = len(losses)
wr           = 100 * sum(1 for g in games if g['outcome'] == 'win') / total
tc_counts    = Counter(g['time_class'] for g in games)

print(f'Total games:  {total}  (Win%: {wr:.1f}%)')
for tc, n in sorted(tc_counts.items()):
    print(f'  {tc}: {n}')
print(f'Total losses: {total_losses}')

## Step 2 — Rating over time

In [ ]:
unique_tc = sorted(tc_counts)
tc_colors = {'bullet': '#c0392b', 'blitz': '#2980b9', 'rapid': '#27ae60'}
fig, axes = plt.subplots(1, max(len(unique_tc), 1), figsize=(7 * len(unique_tc), 4))
if len(unique_tc) == 1:
    axes = [axes]
for ax, tc in zip(axes, unique_tc):
    tc_games = [g for g in games if g['time_class'] == tc and g['my_rating'] > 0]
    if not tc_games: continue
    dates   = [datetime.datetime.fromtimestamp(g['end_time']) for g in tc_games]
    ratings = [g['my_rating'] for g in tc_games]
    col     = tc_colors.get(tc, '#888')
    ax.plot(dates, ratings, linewidth=0.7, alpha=0.8, color=col)
    ax.set_title(f'{USERNAME} — {tc.capitalize()} rating')
    ax.set_ylabel('Rating')
    ax.set_ylim(min(ratings) - 50, max(ratings) + 50)
    ax.tick_params(axis='x', rotation=20)
plt.suptitle(f'{USERNAME}: Rating progression', fontsize=13)
plt.tight_layout()
plt.show()

## Step 3 — Loss distribution by game length

In [ ]:
loss_lengths = [g['halfmoves'] for g in losses]
win_lengths  = [g['halfmoves'] for g in games if g['outcome'] == 'win']

phase_counts = Counter(g['phase'] for g in losses)
print('Losses by phase:')
for phase in PHASES:
    n   = phase_counts[phase]
    pct = 100 * n / total_losses
    print(f'  {phase:30s}: {n:4d}  ({pct:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(loss_lengths, bins=40, color='#c0392b', alpha=0.8, edgecolor='none')
axes[0].axvline(OPENING_END,    color='yellow',     linestyle='--', linewidth=1.5, label='Move 10')
axes[0].axvline(MIDDLEGAME_END, color='lightyellow', linestyle='--', linewidth=1.5, label='Move 25')
axes[0].set_xlabel('Half-moves played')
axes[0].set_ylabel('Losses')
axes[0].set_title(f'{USERNAME}: Loss distribution by game length')
axes[0].legend()

bins = range(0, max(loss_lengths + win_lengths) + 5, 3)
axes[1].hist(win_lengths,  bins=bins, color='#27ae60', alpha=0.6, label='Wins',   edgecolor='none')
axes[1].hist(loss_lengths, bins=bins, color='#c0392b', alpha=0.6, label='Losses', edgecolor='none')
axes[1].set_xlabel('Half-moves played')
axes[1].set_ylabel('Games')
axes[1].set_title('Wins vs Losses by game length')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Median loss length: {np.median(loss_lengths):.0f} half-moves ({np.median(loss_lengths)/2:.0f} full moves)')
print(f'Median win  length: {np.median(win_lengths):.0f} half-moves ({np.median(win_lengths)/2:.0f} full moves)')

## Step 4 — Win rate by game length bucket

In [ ]:
bucket_size  = 10
bucket_stats = defaultdict(Counter)
for g in games:
    b = (g['halfmoves'] // bucket_size) * bucket_size
    bucket_stats[b][g['outcome']] += 1

xs, wrs, totals = [], [], []
for b in sorted(bucket_stats):
    c = bucket_stats[b]
    t = sum(c.values())
    if t < 5: continue
    xs.append(b // 2)
    wrs.append(100 * c['win'] / t)
    totals.append(t)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#27ae60' if wr >= 50 else '#c0392b' for wr in wrs]
bars   = ax.bar(xs, wrs, width=4, color=colors, edgecolor='none', alpha=0.85)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axvline(10, color='yellow',     linestyle=':', linewidth=1.5, label='Move 10')
ax.axvline(25, color='lightyellow', linestyle=':', linewidth=1.5, label='Move 25')
ax.set_xlabel('Full move number (5-move windows)')
ax.set_ylabel('Win %')
ax.set_title(f'{USERNAME}: Win rate by game length')
ax.legend()
for bar, wr, n in zip(bars, wrs, totals):
    if n >= 10:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{wr:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## Step 5 — How do losses end, by phase

In [ ]:
end_types  = ['Checkmate', 'Resignation', 'Timeout', 'Other']
colors_end = ['#c0392b', '#e67e22', '#8e44ad', '#888']
phase_end  = {ph: Counter() for ph in PHASES}
for g in losses:
    phase_end[g['phase']][classify_end(g['end_reason'])] += 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, phase in zip(axes, PHASES):
    c     = phase_end[phase]
    total_ph = sum(c.values())
    ax.bar(end_types, [c[et] for et in end_types], color=colors_end, edgecolor='none')
    ax.set_title(f'{phase}\n({total_ph} losses)')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle(f'{USERNAME}: How do losses end — by phase?', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6 — Breakdown by color and time class

In [ ]:
# By color
print('By color:')
for color in ['white', 'black']:
    closs = [g for g in losses if g['color'] == color]
    if not closs: continue
    print(f'  As {color} ({len(closs)} losses):')
    for phase in PHASES:
        n   = sum(1 for g in closs if g['phase'] == phase)
        pct = 100 * n / len(closs)
        print(f'    {phase:30s}: {n:3d}  ({pct:.0f}%)')

# By time class (only if multiple)
if len(unique_tc) > 1:
    print()
    print('By time class:')
    for tc in unique_tc:
        tc_games  = [g for g in games  if g['time_class'] == tc]
        tc_losses = [g for g in losses if g['time_class'] == tc]
        if not tc_games: continue
        tc_wr = 100 * sum(1 for g in tc_games if g['outcome'] == 'win') / len(tc_games)
        print(f'  {tc.capitalize()} ({len(tc_games)} games, {tc_wr:.1f}% win):')
        for phase in PHASES:
            n   = sum(1 for g in tc_losses if g['phase'] == phase)
            pct = 100 * n / len(tc_losses) if tc_losses else 0
            print(f'    {phase:30s}: {n:3d}  ({pct:.0f}%)')

## Step 7 — Summary

In [ ]:
opening_pct    = 100 * phase_counts[PHASES[0]] / total_losses
middlegame_pct = 100 * phase_counts[PHASES[1]] / total_losses
endgame_pct    = 100 * phase_counts[PHASES[2]] / total_losses
dominant_phase = max(phase_counts, key=phase_counts.get)

print(f'{USERNAME.upper()} — GAME PHASE SUMMARY')
print('=' * 50)
print(f'Total games:  {total}  ({wr:.1f}% win rate)')
print(f'Total losses: {total_losses}')
print()
print(f'  Opening losses:    {phase_counts[PHASES[0]]:4d}  ({opening_pct:.1f}%)')
print(f'  Middlegame losses: {phase_counts[PHASES[1]]:4d}  ({middlegame_pct:.1f}%)')
print(f'  Endgame losses:    {phase_counts[PHASES[2]]:4d}  ({endgame_pct:.1f}%)')
print(f'\nDominant loss phase: {dominant_phase}')
print()
if opening_pct > 35:
    print('>> Opening losses are significant — opening study warranted.')
if middlegame_pct > 40:
    print('>> Middlegame is the main problem — tactical/positional training needed.')
if endgame_pct > 30:
    print('>> Endgame losses are significant — endgame technique study warranted.')
timeout_pct = 100 * sum(1 for g in losses if g['end_reason'] in ('timeout','abandoned')) / total_losses
if timeout_pct > 15:
    print(f'>> Time management issue — {timeout_pct:.0f}% of losses are timeouts/abandoned.')